In [ ]:
# -------------------------------------------------------------
# Common pre‑amble – shared across all notebooks
# -------------------------------------------------------------
from config.notebook_setup import *


# 04 – Model Evaluation

This notebook evaluates all trained models saved in the previous **03_MODEL_TRAINING** step. It computes standard metrics on the test split, generates a confusion matrix and ROC curves for every class, and stores the results on disk so that they can be reused by later notebooks.

## 1. Setup

In [ ]:

import os, json, joblib, itertools, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize
from src.dataset import load_data
from src.utils.model_storage import list_available_models, load_model

warnings.filterwarnings('ignore')
%matplotlib inline


## 2. Load data and models

In [ ]:

# ---- Parameters ----
N_CLASSES   = 10    # must match training
RESULTS_DIR = pathlib.Path(f'experiment_with_{N_CLASSES}_classes/results')
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

# Load test split
X_train, y_train, X_test, y_test, label_names = load_data(N_CLASSES)
print(f'Test documents: {len(X_test):,}')

# Discover models
available = list_available_models(N_CLASSES)
model_names = available.get(N_CLASSES, [])
if not model_names:
    raise RuntimeError('No trained models found. Run 03_MODEL_TRAINING first.')
print('Models found:', model_names)


## 3. Evaluate each model

In [ ]:

metrics_all = {}
y_true_bin = label_binarize(y_test, classes=label_names)
for mname in model_names:
    print(f'\n### Evaluating {mname}')
    model, train_metrics = load_model(mname, N_CLASSES)
    y_pred = model.predict(X_test)
    # Probabilities / decision scores for ROC
    if hasattr(model, 'predict_proba'):
        y_scores = model.predict_proba(X_test)
    elif hasattr(model, 'decision_function'):
        y_scores = model.decision_function(X_test)
        # SVM decision_function can be shape (n_samples, n_classes)
        # Convert to probabilities via softmax for AUC
        import scipy.special as sp
        y_scores = sp.softmax(y_scores, axis=1)
    else:
        y_scores = None

    acc  = accuracy_score(y_test, y_pred)
    f1m  = f1_score(y_test, y_pred, average='macro')
    f1w  = f1_score(y_test, y_pred, average='weighted')
    report = classification_report(y_test, y_pred, target_names=label_names, output_dict=True)
    cm = confusion_matrix(y_test, y_pred, labels=label_names)

    mdict = {'accuracy': acc, 'f1_macro': f1m, 'f1_weighted': f1w, 'classification_report': report}
    metrics_all[mname] = mdict

    # ---- Save confusion matrix ----
    fig, ax = plt.subplots(figsize=(6,6))
    im = ax.imshow(cm, interpolation='nearest')
    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    ax.set_xticklabels(label_names, rotation=90)
    ax.set_yticklabels(label_names)
    ax.set_title(f'Confusion matrix – {mname}')
    plt.colorbar(im)
    plt.tight_layout()
    fig_path = RESULTS_DIR / f'confusion_{mname.replace(" ", "_")}.png'
    fig.savefig(fig_path, dpi=150)
    plt.close(fig)

    # ---- ROC curve ----
    if y_scores is not None:
        fig, ax = plt.subplots()
        for i, class_name in enumerate(label_names):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
            auc = roc_auc_score(y_true_bin[:, i], y_scores[:, i])
            ax.plot(fpr, tpr, label=f'{class_name} (AUC={auc:.2f})')
        ax.plot([0,1], [0,1], linestyle='--')
        ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
        ax.set_title(f'ROC curves – {mname}')
        ax.legend(loc='lower right')
        roc_path = RESULTS_DIR / f'roc_{mname.replace(" ", "_")}.png'
        fig.savefig(roc_path, dpi=150)
        plt.close(fig)

    # ---- Worst classified docs ----
    wrong_idx = [i for i,(true,pred) in enumerate(zip(y_test, y_pred)) if true!=pred]
    worst_df = pd.DataFrame({
        'doc_id': wrong_idx,
        'true': [y_test[i] for i in wrong_idx],
        'pred': [y_pred[i] for i in wrong_idx],
        'text': [X_test[i][:500] for i in wrong_idx]  # truncated preview
    })
    worst_path = RESULTS_DIR / f'worst_{mname.replace(" ", "_")}.csv'
    worst_df.to_csv(worst_path, index=False)

# ---- Save metrics summary ----
metrics_json = RESULTS_DIR / 'test_metrics.json'
with open(metrics_json, 'w') as f:
    json.dump(metrics_all, f, indent=2)
print(f'All metrics saved to {metrics_json}')
